## Redis — Deep Dive

Source: https://www.hellointerview.com/learn/system-design/deep-dives/redis

Redis is an **in-memory, single-threaded data structure store** written in C.
It prioritises speed over durability — reads take microseconds and it handles ~100k writes/sec.

> Key insight for system design interviews: Redis is not just a cache.
> It is a versatile data structure store with distinct solutions for caching,
> locking, rate limiting, leaderboards, pub/sub, and more.


### 1. Core Architecture

| Property | Detail |
|---|---|
| Storage | In-memory (RAM) |
| Threading | Single-threaded |
| Language | C |
| Write throughput | ~100,000 ops/sec |
| Read latency | Microsecond range |
| Key type | Always a string |
| Value type | One of 8 data structures |

**Why single-threaded?**
No locks needed — commands execute atomically one at a time.
This is why Lua scripts work as distributed atomic operations (see rate limiting notebook).

---

### 2. Data Structures

Each structure is purpose-built for a specific class of problem.

#### Strings
The simplest type. Value is a byte sequence — can hold text, integers, or binary.
```
SET  user:1:name  "alice"
GET  user:1:name          → "alice"
INCR user:1:visits        → atomic increment (no race condition)
EXPIRE user:1:session 300 → TTL in seconds
```
**Use cases:** caching, session tokens, counters, rate limiting counters.

---

#### Hashes
A map of field → value stored under one key. Like a Python `dict` as a Redis value.
```
HSET  product:1  name "iPhone"  price "999"  stock "50"
HGET  product:1  price          → "999"
HGETALL product:1               → {name, price, stock}
HINCRBY product:1 stock -1      → atomic decrement (inventory)
```
**Use cases:** storing objects/entities without serialising to JSON, partial field updates.

---

#### Lists
Ordered sequence with O(1) push/pop from both ends. Backed by a doubly-linked list.
```
LPUSH  queue:jobs  "job3"    → push to left (head)
RPUSH  queue:jobs  "job4"    → push to right (tail)
LPOP   queue:jobs            → pop from left
RPOP   queue:jobs            → pop from right
LRANGE queue:jobs 0 -1       → all elements
```
**Use cases:** FIFO queues, activity feeds, recent items lists.

---

#### Sets
Unordered collection of unique strings. O(1) add/remove/membership.
```
SADD   user:1:tags  "python"  "redis"
SADD   user:2:tags  "redis"   "go"
SINTER user:1:tags user:2:tags  → {"redis"}   (intersection)
SUNION user:1:tags user:2:tags  → {"python", "redis", "go"}
SISMEMBER user:1:tags "python"  → 1 (true)
```
**Use cases:** unique visitor tracking, tagging, friend/follower graphs, deduplication.

---

#### Sorted Sets (ZSets)
Every member has a float **score**. Members are ordered by score. O(log N) operations.
```
ZADD  leaderboard  1500 "alice"
ZADD  leaderboard  2300 "bob"
ZADD  leaderboard  1900 "carol"
ZRANK leaderboard "alice"           → 0  (rank from lowest)
ZREVRANK leaderboard "bob"          → 0  (rank from highest)
ZRANGE leaderboard 0 -1 WITHSCORES → all members by score
ZINCRBY leaderboard 100 "alice"     → atomic score update
```
**Use cases:** leaderboards, priority queues, sliding window rate limiting (score = timestamp).

---

#### Bloom Filters
Probabilistic data structure — answers "is this element possibly in the set?" in O(1).
- **False positives possible** — may say "yes" when the answer is "no"
- **False negatives impossible** — if it says "no", it is definitely not in the set
```
BF.ADD   seen:urls  "https://example.com"
BF.EXISTS seen:urls "https://example.com"  → 1 (probably yes)
BF.EXISTS seen:urls "https://unknown.com"  → 0 (definitely no)
```
**Use cases:** web crawler deduplication, spam filter, "have you seen this notification?"

---

#### Geospatial Index
Stores lat/lon coordinates and supports radius queries. Internally backed by a sorted set.
```
GEOADD  locations  -122.4  37.7  "san_francisco"
GEOADD  locations  -118.2  34.0  "los_angeles"
GEODIST locations san_francisco los_angeles km  → ~559 km
GEORADIUS locations -122.4 37.7 100 km          → nearby locations
```
**Use cases:** "find nearby drivers/restaurants", proximity search, geo-fencing.

---

#### Streams
Append-only log of events. Similar to Kafka but simpler. Each entry has a unique ID.
```
XADD  events  *  user_id 42  action "purchase"
XREAD COUNT 10 STREAMS events 0    → read from beginning
XREAD BLOCK 0 STREAMS events $    → block until new event (real-time)
```
**Use cases:** event sourcing, activity feeds, real-time analytics pipelines.

---

### 3. Deployment Models

#### Single Node
One Redis process. Simple but no fault tolerance.
```
Client → Redis (single node)
```

#### High Availability (Primary + Replica)
One primary handles writes. One or more replicas sync asynchronously.
Redis Sentinel monitors health and promotes a replica if the primary dies.
```
Client → Primary (writes)
       → Replica 1 (reads)  ← async replication from primary
       → Replica 2 (reads)
Sentinel watches all three — promotes replica on primary failure
```

#### Cluster Mode
Data is sharded across multiple nodes using **16,384 hash slots**.
Each key is assigned to a slot: `slot = CRC16(key) % 16384`
Each node owns a range of slots.

```
Node A: slots 0    – 5460     (keys hashing here → Node A)
Node B: slots 5461 – 10922    (keys hashing here → Node B)
Node C: slots 10923 – 16383   (keys hashing here → Node C)
```

Clients cache the slot-to-node mapping and connect directly to the correct node.
If a key moves (resharding), Redis returns a `MOVED` redirect and the client updates its map.

**Critical constraint:** All keys in a multi-key operation must live on the same node.
Use **hash tags** `{user:1}:session` and `{user:1}:cart` to force co-location.

---

### 4. Persistence

Redis is in-memory but offers two persistence mechanisms:

#### RDB (Redis Database Snapshot)
Point-in-time snapshots written to disk at configurable intervals.
- Fast restarts (load one file)
- Risk: data since last snapshot is lost on crash

#### AOF (Append-Only File)
Every write command is appended to a log file. On restart, Redis replays the log.
- `appendfsync always` — fsync after every command (safest, slowest)
- `appendfsync everysec` — fsync every second (good balance)
- `appendfsync no` — OS decides (fastest, least safe)
- Risk: max 1 second of data loss with `everysec`

#### AWS MemoryDB
If you need Redis API + true durability (database-level commit guarantees), use AWS MemoryDB.
It writes to a multi-AZ transaction log before acknowledging — no data loss on failure.

---

### 5. Primary Use Cases

#### Caching
```
GET key → hit: return value
        → miss: fetch from DB, SET key value EX ttl, return value
```
Combine with TTL-based eviction and a cache eviction policy (LRU, LFU).
Hot key problem: use key replication, local L1 cache, or virtual shards (see caching notebook).

---

#### Distributed Locking
Single instance — use `SET key value NX PX timeout`:
- `NX` — only set if key does not exist (atomic acquire)
- `PX` — auto-expire in milliseconds (prevent deadlock if holder crashes)
```
SET lock:resource "owner-uuid" NX PX 5000
→ OK   (lock acquired)
→ nil  (already locked)
```

**Redlock** — for locking across N independent Redis instances (no replication):
1. Acquire lock on majority (N/2 + 1) instances within a timeout
2. If acquired on majority, lock is held
3. Release on all instances when done
Handles partial failures — if one node dies, the lock still holds on the majority.

---

#### Leaderboards
Sorted sets are a perfect fit — O(log N) insert and rank query.
```
ZADD   scores  1500 "alice"
ZINCRBY scores 200  "alice"    → alice now has 1700
ZREVRANK scores "alice"        → rank from top (0-indexed)
ZREVRANGE scores 0 9 WITHSCORES → top 10
```

---

#### Rate Limiting
Fixed window with INCR + EXPIRE (see rate limiting notebook for full implementations):
```lua
local count = redis.call('INCR', key)
if count == 1 then redis.call('EXPIRE', key, window) end
if count > limit then return 0 end
return 1
```

Sliding window log with sorted sets — score = timestamp, ZREMRANGEBYSCORE evicts old entries.

---

#### Pub/Sub
Fire-and-forget messaging. No persistence — if a subscriber is offline, messages are lost.
```
SUBSCRIBE  channel:news      ← subscriber listens
PUBLISH    channel:news "breaking update"  ← publisher sends
```
**Use cases:** real-time notifications, live dashboards, chat.
**Limitation:** no delivery guarantee, no replay.

#### Streams (durable Pub/Sub)
Persistent ordered log with consumer groups — closer to Kafka.
```
XADD events * type "order_placed" order_id "42"
XREADGROUP GROUP workers consumer1 COUNT 1 STREAMS events >
XACK  events workers <message-id>   ← acknowledge processed
```
**Use cases:** event sourcing, audit logs, durable task queues.

---

### 6. Limitations

| Limitation | Detail |
|---|---|
| Memory bound | All data must fit in RAM |
| No durability guarantee | AOF gives ~1s loss; RDB more |
| Single-node ops only | Multi-key ops require same node |
| Hot key | No built-in solution — application must handle |
| No relational queries | Not a replacement for Postgres |
| Single-threaded | CPU-bound workloads can bottleneck |

---

### 7. Quick Reference — When to use Redis

| Problem | Redis solution |
|---|---|
| Cache with TTL | String + EXPIRE |
| Store an object | Hash |
| FIFO queue | List (LPUSH/RPOP) |
| Unique visitor count | Set (SADD + SCARD) |
| Leaderboard / ranking | Sorted Set |
| Rate limiting | String INCR + EXPIRE / ZSet |
| Distributed lock | SET NX PX / Redlock |
| "Have I seen this?" | Bloom Filter |
| Nearby search | Geospatial Index |
| Durable event stream | Streams |
| Real-time notifications | Pub/Sub |


---

### 8. Summary

#### Pros

| # | Pro | Detail |
|---|---|---|
| 1 | **High Performance** | Sub-millisecond latency, 100k+ writes/sec — ideal for real-time applications |
| 2 | **Versatile Data Structures** | Strings, Hashes, Sets, Sorted Sets, Streams, Geospatial — one tool, many problems |
| 3 | **Ease of Use** | Simple, intuitive commands — low learning curve |
| 4 | **Scalability** | Horizontal scaling via Cluster (16,384 hash slots) + read scaling via replicas |
| 5 | **Durability Options** | RDB snapshots + AOF log cover most durability needs without sacrificing speed |
| 6 | **Wide Applications** | Caching, sessions, analytics, leaderboards, rate limiting, pub/sub, locking |

---

#### Cons

| # | Con | Detail |
|---|---|---|
| 1 | **Memory-Intensive** | All data lives in RAM — expensive at large scale |
| 2 | **Potential Data Loss** | Without persistence, crashes lose all data; AOF still risks ~1s loss |
| 3 | **Manual Memory Management** | Must configure eviction policies (LRU, LFU) to handle memory overflow |
| 4 | **Limited Query Capabilities** | No joins, no relational queries — not a replacement for PostgreSQL |
| 5 | **Single-node constraint** | Multi-key ops require all keys on same node — key design is critical |

---

#### Practical Use Cases

| Use Case | Industry Example | Redis Implementation |
|---|---|---|
| **Caching** | E-commerce caches product details to cut DB load and improve page load | `SET product:1 <json> EX 300` → `GET product:1` |
| **Session Management** | Gaming platforms store user sessions to keep app servers stateless | `HSET session:<token> user_id 42` + `EXPIRE session:<token> 1800` |
| **Real-Time Analytics** | Retailers track page views and click events for instant dashboards | `XADD events * page /home user_id 42` → `XREAD` by workers |
| **Leaderboards** | Games rank players by score and show top-N | `ZADD scores 1500 alice` → `ZREVRANGE scores 0 9 WITHSCORES` → `ZREMRANGEBYRANK` to prune |
| **Rate Limiting** | API gateways cap requests per user per window | `INCR rate:user:42` + `EXPIRE` — reject if over limit |
| **Proximity Search** | Ride-sharing finds the nearest available drivers | `GEOADD drivers lng lat driver:1` → `GEORADIUS drivers lng lat 5 km` |
| **Event Sourcing / Queues** | Chat apps deliver messages; workers process tasks with failure recovery | `PUBLISH channel msg` for fire-and-forget; `XCLAIM` in streams for at-least-once delivery |
| **Fraud Detection** | Banks flag suspicious activity in real time before a transaction completes | Aggregate recent transactions in a Hash or Stream; threshold check in Lua script |

---

> Redis is a powerful tool for any problem requiring **low-latency data access at scale**.
> Its versatility across retail, gaming, ride-hailing, and finance makes it one of the
> most important components to understand for system design interviews.


---

### 9. Redis Pub/Sub and Streams — Deep Dive

Both solve the same high-level problem — broadcasting messages from producers to consumers —
but they make very different trade-offs.

---

#### Pub/Sub

**How it works**

Publishers send messages to a **channel**. All active subscribers on that channel receive
the message instantly. Redis acts purely as a broker — it does not store messages.

```
Publisher                  Redis Broker              Subscribers
─────────                  ────────────              ───────────
PUBLISH news "update"  →   channel: news   →  SUBSCRIBE news (client A)
                                            →  SUBSCRIBE news (client B)
                                            →  SUBSCRIBE news (client C)
```

**Core commands**

```
# Subscriber side
SUBSCRIBE  channel:orders           ← listen to one channel
PSUBSCRIBE order:*                  ← pattern match (glob) — all channels starting with "order:"

# Publisher side
PUBLISH  channel:orders  "order_placed:42"

# Inspect
PUBSUB CHANNELS          ← list all active channels
PUBSUB NUMSUB channel    ← count subscribers on a channel
```

**Message delivery model**

```
fire-and-forget
      ↓
message sent → delivered to all CURRENTLY connected subscribers
             → if subscriber is offline: message is LOST permanently
             → no acknowledgement, no retry, no replay
```

**Pattern subscriptions (PSUBSCRIBE)**

```
PSUBSCRIBE order:*
→ receives messages from: order:placed, order:shipped, order:cancelled
→ message includes the matched channel name so the handler knows the source
```

**What happens when a subscriber disconnects**

```
t=0   Client A subscribes to "news"
t=1   Publisher sends 5 messages
t=2   Client A disconnects
t=3   Publisher sends 5 more messages   ← Client A misses these forever
t=4   Client A reconnects               ← starts from NOW, no backfill
```

**When to use Pub/Sub**

- Real-time notifications where slight message loss is acceptable
- Live dashboards — dashboard refreshes periodically anyway
- Chat systems where offline users are handled by push notifications separately
- Cache invalidation signals — tell other nodes to evict a key
- Presence updates — user is typing, user went online

**Limitations**

| Limitation | Impact |
|---|---|
| No persistence | Offline subscribers lose messages |
| No acknowledgement | No way to confirm delivery |
| No replay | Cannot re-read past messages |
| No consumer groups | All subscribers get every message (fan-out only) |
| Memory pressure | Slow subscriber buffers fill up → Redis drops messages |

---

#### Streams

Redis Streams is a persistent, ordered, append-only log — closer to Apache Kafka than Pub/Sub.
Every message is stored and can be re-read by any consumer at any time.

**How it works**

```
Producers append entries to the stream.
Each entry gets a unique ID: <millisecondsTimestamp>-<sequenceNumber>
e.g. "1713001234567-0"

Consumers read from any position in the stream.
Consumer Groups allow multiple workers to divide the work.
```

**Core commands**

```
# Producer
XADD  stream:orders  *  order_id 42  user_id 7  total 99.99
#                    ↑ auto-generate ID

# Consumer — simple read
XREAD COUNT 10 STREAMS stream:orders 0     ← read from beginning
XREAD COUNT 10 STREAMS stream:orders $     ← read only new messages (like tail -f)
XREAD BLOCK 0  STREAMS stream:orders $     ← block until new message arrives

# Range queries
XRANGE stream:orders - +                   ← all entries
XRANGE stream:orders 1713001234567-0 +     ← from a specific ID onwards
XREVRANGE stream:orders + - COUNT 5        ← last 5 entries

# Stream metadata
XLEN    stream:orders                      ← total entry count
XINFO STREAM stream:orders                 ← detailed stream info
```

**Consumer Groups — work distribution**

Consumer groups allow multiple workers to divide a stream's messages — each message
goes to exactly one worker in the group. This is the key difference from Pub/Sub's fan-out.

```
                         stream:orders
                         ─────────────
                         msg-1  msg-2  msg-3  msg-4  msg-5

Consumer Group "workers"
  ├── worker-1  receives: msg-1, msg-3, msg-5
  └── worker-2  receives: msg-2, msg-4
```

```
# Create group (start from beginning "0" or latest "$")
XGROUP CREATE stream:orders workers 0

# Worker reads its share (> means undelivered messages only)
XREADGROUP GROUP workers worker-1 COUNT 5 STREAMS stream:orders >

# Acknowledge processed message (removes from pending list)
XACK stream:orders workers <message-id>

# Inspect pending (unacknowledged) messages
XPENDING stream:orders workers - + 10
```

**Failure recovery with XCLAIM**

If a worker crashes mid-processing, its messages stay in the Pending Entry List (PEL).
Another worker can claim them after a timeout:

```
# Find messages pending > 30 seconds (stuck due to crashed worker)
XPENDING stream:orders workers - + 10

# Claim ownership of a stuck message
XCLAIM stream:orders workers worker-2 30000 <message-id>
#                                     ↑ min idle time in ms
```

**Stream trimming — managing memory**

Streams grow forever unless trimmed. Two strategies:

```
# Keep only the last 1000 entries (exact)
XTRIM stream:orders MAXLEN 1000

# Keep approximately 1000 entries (faster, uses ~ prefix)
XTRIM stream:orders MAXLEN ~ 1000

# Auto-trim on XADD
XADD stream:orders MAXLEN ~ 1000  *  order_id 42  ...
```

---

#### Pub/Sub vs Streams — side-by-side

| Property | Pub/Sub | Streams |
|---|---|---|
| Persistence | No — fire and forget | Yes — stored until trimmed |
| Offline consumers | Messages lost | Messages waiting when they reconnect |
| Replay | No | Yes — read from any past ID |
| Consumer groups | No — all subscribers get all messages | Yes — work is divided across workers |
| Acknowledgement | No | Yes — XACK removes from pending list |
| Failure recovery | No | Yes — XCLAIM reassigns stuck messages |
| Message ordering | Per-channel FIFO | Global FIFO within stream |
| Throughput | Extremely high (no disk) | High (append-only log) |
| At-Most Once | ✅ only option | ✅ read without ACK |
| At-Least Once | ❌ no ACK path | ✅ XACK + XCLAIM |
| Exactly Once | ❌ | ⚠️ manual idempotency key |
| Use case | Ephemeral notifications | Durable event log, task queues |

---

#### Pub/Sub vs Streams vs Kafka

| | Redis Pub/Sub | Redis Streams | Apache Kafka |
|---|---|---|---|
| Persistence | None | Yes (in-memory) | Yes (disk, long-term) |
| Retention | None | Short (RAM limited) | Days/weeks/forever |
| Throughput | Highest | High | Very high |
| Consumer groups | No | Yes | Yes |
| Replay | No | Yes (within retention) | Yes (full history) |
| Ordering | Per channel | Per stream | Per partition |
| Acknowledgement | No | Yes — XACK | Yes — offset commit |
| Failure recovery | No | Yes — XCLAIM | Yes — re-read from offset |
| **At-Most Once** | ✅ only option | ✅ read without ACK | ✅ auto-commit offsets |
| **At-Least Once** | ❌ no ACK path | ✅ XACK + XCLAIM | ✅ manual offset commit |
| **Exactly Once** | ❌ | ⚠️ manual idempotency key | ✅ native transactions |
| Setup complexity | Zero | Zero | High |
| Best for | Real-time signals | Short-lived task queues | Durable event sourcing |

**Delivery semantics explained:**

| Semantic | Data Loss | Duplicates | How |
|---|---|---|---|
| At-Most Once | Possible | Never | Send and forget — no ACK, no retry |
| At-Least Once | Never | Possible | Retry until ACK — consumer must be idempotent |
| Exactly Once | Never | Never | Idempotent producer + transactional consumer |

**Rule of thumb:**
- Use **Pub/Sub** when you need instant delivery and can tolerate loss → at-most once is fine
- Use **Streams** when you need durability and at-least once → add idempotency key for exactly once
- Use **Kafka** when you need long-term retention, cross-service replay, or native exactly-once guarantees

---

#### Common Patterns

**1. Chat room (Pub/Sub)**
```
Each room = one channel
User joins room  → SUBSCRIBE room:42
User sends msg   → PUBLISH room:42 "hello"
User leaves room → UNSUBSCRIBE room:42
```

**2. Cache invalidation broadcast (Pub/Sub)**
```
Service writes new value to DB
→ PUBLISH cache:invalidate "product:1"
→ All app servers receive signal and evict "product:1" from local L1 cache
```

**3. Distributed task queue (Streams + Consumer Groups)**
```
API server  → XADD jobs * type "resize_image" url "..."
Worker pool → XREADGROUP GROUP workers workerN COUNT 1 STREAMS jobs >
            → process task
            → XACK jobs workers <id>
Monitor     → XPENDING jobs workers ... detects stuck tasks → XCLAIM
```

**4. Activity feed (Streams)**
```
User actions  → XADD feed:user:42 * action "liked" post_id "99"
Feed reader   → XREVRANGE feed:user:42 + - COUNT 20   ← last 20 actions
Trim old data → XTRIM feed:user:42 MAXLEN ~ 1000
```


In [ ]:
"""
Redis Pub/Sub and Streams — simulated implementations.

In production replace the in-memory broker/stream with:
  import redis
  r = redis.Redis(host="localhost", port=6379, decode_responses=True)
"""
import time
import threading
from collections import defaultdict, deque
from dataclasses import dataclass, field


# ─────────────────────────────────────────────
# Simulated Redis Pub/Sub
# ─────────────────────────────────────────────
class PubSubBroker:
    """
    Fire-and-forget broker.
    Messages are delivered only to subscribers connected at publish time.
    Offline subscribers miss messages permanently.
    """
    def __init__(self):
        self._channels: dict[str, list] = defaultdict(list)   # channel → [callbacks]
        self._lock = threading.Lock()

    def subscribe(self, channel: str, callback) -> None:
        with self._lock:
            self._channels[channel].append(callback)
        print(f"  [SUBSCRIBE] {callback.__self__.name} → {channel}")

    def unsubscribe(self, channel: str, callback) -> None:
        with self._lock:
            self._channels[channel].remove(callback)

    def publish(self, channel: str, message: str) -> int:
        with self._lock:
            subscribers = list(self._channels.get(channel, []))
        for cb in subscribers:
            cb(channel, message)
        print(f"  [PUBLISH]   {channel} → '{message}'  ({len(subscribers)} subscribers)")
        return len(subscribers)


class Subscriber:
    def __init__(self, name: str):
        self.name = name
        self.received: list[tuple[str, str]] = []

    def on_message(self, channel: str, message: str) -> None:
        self.received.append((channel, message))
        print(f"    [{self.name}] received on '{channel}': '{message}'")


# ─────────────────────────────────────────────
# Simulated Redis Streams
# ─────────────────────────────────────────────
@dataclass
class StreamEntry:
    id: str
    data: dict


class RedisStream:
    """
    Append-only log with consumer groups, acknowledgement, and XCLAIM.
    """
    def __init__(self, name: str):
        self.name = name
        self._entries: list[StreamEntry] = []
        self._groups: dict[str, dict] = {}   # group → {last_id, pending: {consumer: [ids]}}
        self._lock = threading.Lock()
        self._seq = 0

    def xadd(self, **fields) -> str:
        with self._lock:
            entry_id = f"{int(time.time() * 1000)}-{self._seq}"
            self._seq += 1
            self._entries.append(StreamEntry(id=entry_id, data=fields))
        print(f"  [XADD]  {self.name}  id={entry_id}  data={fields}")
        return entry_id

    def xread(self, count: int = 10, last_id: str = "0") -> list[StreamEntry]:
        with self._lock:
            result = [e for e in self._entries if e.id > last_id]
        return result[:count]

    def xrange(self, count: int = 10) -> list[StreamEntry]:
        with self._lock:
            return list(self._entries[-count:])

    def xgroup_create(self, group: str, start_id: str = "0") -> None:
        with self._lock:
            self._groups[group] = {
                "last_id": start_id,
                "pending": defaultdict(list),   # consumer → [entry_ids]
                "pending_time": {},             # entry_id → time delivered
            }
        print(f"  [XGROUP CREATE] group='{group}' start='{start_id}'")

    def xreadgroup(self, group: str, consumer: str, count: int = 1) -> list[StreamEntry]:
        with self._lock:
            g = self._groups[group]
            last = g["last_id"]
            undelivered = [e for e in self._entries if e.id > last][:count]
            for e in undelivered:
                g["last_id"] = e.id
                g["pending"][consumer].append(e.id)
                g["pending_time"][e.id] = time.monotonic()
        for e in undelivered:
            print(f"  [XREADGROUP] group={group} consumer={consumer} → id={e.id} data={e.data}")
        return undelivered

    def xack(self, group: str, consumer: str, entry_id: str) -> None:
        with self._lock:
            g = self._groups[group]
            if entry_id in g["pending"][consumer]:
                g["pending"][consumer].remove(entry_id)
                g["pending_time"].pop(entry_id, None)
        print(f"  [XACK]  group={group} consumer={consumer} id={entry_id}")

    def xclaim(self, group: str, new_consumer: str, min_idle_ms: int) -> list[str]:
        """Reassign messages idle longer than min_idle_ms to new_consumer."""
        claimed = []
        now = time.monotonic()
        with self._lock:
            g = self._groups[group]
            for consumer, ids in list(g["pending"].items()):
                for entry_id in list(ids):
                    idle_ms = (now - g["pending_time"].get(entry_id, now)) * 1000
                    if idle_ms >= min_idle_ms and consumer != new_consumer:
                        ids.remove(entry_id)
                        g["pending"][new_consumer].append(entry_id)
                        g["pending_time"][entry_id] = now
                        claimed.append(entry_id)
        if claimed:
            print(f"  [XCLAIM] → {new_consumer} claimed {claimed}")
        return claimed

    def xpending(self, group: str) -> dict:
        with self._lock:
            return {c: list(ids) for c, ids in self._groups[group]["pending"].items() if ids}

    def xtrim(self, maxlen: int) -> int:
        with self._lock:
            removed = max(0, len(self._entries) - maxlen)
            self._entries = self._entries[-maxlen:]
        print(f"  [XTRIM]  kept last {maxlen} entries (removed {removed})")
        return removed


# ─────────────────────────────────────────────────────────────────
# Demo 1 — Pub/Sub: chat room + offline subscriber misses messages
# ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("DEMO 1 — Pub/Sub: chat room")
print("=" * 60)

broker = PubSubBroker()
alice = Subscriber("alice")
bob   = Subscriber("bob")

broker.subscribe("room:42", alice.on_message)
broker.subscribe("room:42", bob.on_message)

broker.publish("room:42", "hello everyone")
broker.publish("room:42", "how are you?")

print("\n  [bob goes offline]")
broker.unsubscribe("room:42", bob.on_message)

broker.publish("room:42", "bob will miss this")

print(f"\n  alice received {len(alice.received)} messages")
print(f"  bob   received {len(bob.received)} messages  ← missed 1 after disconnect")

# ─────────────────────────────────────────────────────────────────
# Demo 2 — Streams: task queue with consumer group + failure recovery
# ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("DEMO 2 — Streams: task queue with XCLAIM recovery")
print("=" * 60)

stream = RedisStream("jobs")
stream.xgroup_create("workers", start_id="0")

print("\n  --- Producer adds 4 jobs ---")
stream.xadd(type="resize_image", url="img1.jpg")
stream.xadd(type="resize_image", url="img2.jpg")
stream.xadd(type="send_email",   to="alice@example.com")
stream.xadd(type="send_email",   to="bob@example.com")

print("\n  --- worker-1 picks up 2 jobs ---")
jobs = stream.xreadgroup("workers", "worker-1", count=2)

print("\n  --- worker-1 acks job 1, then crashes before acking job 2 ---")
stream.xack("workers", "worker-1", jobs[0].id)
# worker-1 crashes — jobs[1].id stays in pending

print("\n  --- worker-2 picks up remaining jobs ---")
stream.xreadgroup("workers", "worker-2", count=2)

print(f"\n  Pending entries: {stream.xpending('workers')}")

print("\n  --- Monitor detects stuck job, XCLAIM to worker-2 ---")
stream.xclaim("workers", "worker-2", min_idle_ms=0)

print(f"\n  Pending after XCLAIM: {stream.xpending('workers')}")

# ─────────────────────────────────────────────────────────────────
# Demo 3 — Streams: activity feed with trimming
# ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("DEMO 3 — Streams: activity feed with XTRIM")
print("=" * 60)

feed = RedisStream("feed:user:42")
actions = ["liked post:1", "commented post:2", "shared post:3", "liked post:4", "followed user:7"]
for action in actions:
    feed.xadd(action=action)

print(f"\n  Stream length before trim: {len(feed._entries)}")
feed.xtrim(maxlen=3)
print(f"  Stream length after  trim: {len(feed._entries)}")
print(f"  Remaining entries:")
for e in feed.xrange():
    print(f"    {e.id}  {e.data}")


---

### 10. Message Delivery Semantics

Every messaging system makes a contract about how many times a message will be delivered.
There are three guarantees — they trade off between data loss and duplicate processing.

---

#### At-Most Once

Message is delivered **zero or one time**. It is sent once and never retried.
If the consumer crashes before processing, the message is **lost forever**.

```
Producer → sends message → Consumer
                               ↓
                           crashes before processing
                               ↓
                           message gone — never retried
```

**How it works:**
- Message is removed from the broker as soon as it is sent (no acknowledgement required)
- No retry logic, no persistent queue

**Trade-off:**
- Zero duplicate processing ✅
- Possible data loss ❌

**When to use:**
- Metrics, telemetry, analytics events where losing a few data points is acceptable
- Log shipping where approximate counts are fine
- Notifications where a missed one is tolerable (e.g. "user is typing")

---

#### At-Least Once

Message is delivered **one or more times**. The broker retries until the consumer acknowledges.
If the consumer crashes after processing but before acknowledging, it gets the message again.

```
Producer → sends message → Consumer
                               ↓
                           processes message
                               ↓
                           crashes BEFORE sending ACK
                               ↓
                           Broker retries → Consumer gets message again
                               ↓
                           processed TWICE  ← duplicate
```

**How it works:**
- Broker keeps message in a pending/unacknowledged state until ACK arrives
- If ACK doesn't arrive within timeout → redeliver
- Consumer must be **idempotent** — processing the same message twice must be safe

**Trade-off:**
- No data loss ✅
- Possible duplicate processing ❌ (consumer must handle it)

**Making consumers idempotent:**
```
On receiving message with id="order-42":
  if already_processed("order-42"):
      return   ← skip duplicate
  process_order(42)
  mark_processed("order-42")
```

**When to use:**
- Order processing, payment events — losing a message is worse than processing twice
- Email sending (deduplication via message ID)
- Most production systems default to this

---

#### Exactly Once

Message is delivered **exactly one time** — no loss, no duplicates.
The hardest guarantee to achieve, requiring coordination between producer, broker, and consumer.

```
Producer → sends message (with unique ID)
               ↓
           Broker stores atomically
               ↓
           Consumer processes + ACKs atomically
               ↓
           Even on retry: message processed exactly once
```

**How it works — two-phase approach:**
1. **Idempotent producer** — broker deduplicates if producer sends the same message twice
2. **Transactional consumer** — processing + ACK happen in the same atomic transaction

```
BEGIN TRANSACTION
  result = process(message)
  write result to DB
  ACK message to broker
COMMIT
→ if crash anywhere: transaction rolls back, message is redelivered
→ but broker tracks producer sequence numbers and deduplicates retries
```

**Trade-off:**
- No data loss ✅
- No duplicates ✅
- Higher latency and complexity ❌

**When to use:**
- Financial transactions, inventory deductions — duplicates cause real money loss
- Ledger systems, audit logs requiring exactly one entry per event

---

#### Which system supports which guarantee?

| | At-Most Once | At-Least Once | Exactly Once |
|---|---|---|---|
| **Redis Pub/Sub** | ✅ (only option — fire-and-forget) | ❌ no ACK, no retry | ❌ |
| **Redis Streams** | ✅ (read without ACK) | ✅ XACK + XCLAIM retry | ⚠️ with idempotent consumer + Lua |
| **Apache Kafka** | ✅ (auto-commit offsets) | ✅ manual offset commit | ✅ idempotent producer + transactions |

---

#### How each system implements each guarantee

**Redis Pub/Sub — At-Most Once only**
```
PUBLISH channel message
→ delivered to connected subscribers instantly
→ no ACK path exists — once sent, it's gone
→ at-most once is the only possible contract
```

**Redis Streams — At-Least Once (default)**
```
XREADGROUP GROUP workers consumer1 COUNT 1 STREAMS jobs >
  → message enters Pending Entry List (PEL)
  → worker processes message
  → XACK jobs workers <id>   ← removes from PEL

If worker crashes before XACK:
  → message stays in PEL
  → monitor runs XCLAIM after timeout
  → message redelivered to another worker
  → at-least once: possible duplicate if crash after process but before XACK
```

**Redis Streams — Exactly Once (with idempotency)**
```
On receiving message:
  if redis.EXISTS(f"processed:{message_id}"):
      XACK stream group message_id
      return   ← skip duplicate safely

  process(message)
  redis.SET(f"processed:{message_id}", 1, EX=86400)   ← idempotency key
  XACK stream group message_id
```

**Kafka — Exactly Once (native)**
```
Producer:
  enable.idempotence = true          ← broker deduplicates producer retries
  transactional.id = "producer-1"    ← enables transactions

Consumer:
  isolation.level = read_committed   ← only reads committed transactions
  manual offset commit inside transaction
→ process + commit offset = one atomic operation
```

---

#### Summary

```
At-Most Once:   fast, simple, lossy
                → use for metrics, telemetry, "user is typing"

At-Least Once:  reliable, possible duplicates → make consumers idempotent
                → use for most production systems (orders, emails, jobs)

Exactly Once:   no loss, no duplicates, highest complexity and latency
                → use for financial transactions, ledgers, inventory
```

| Guarantee | Data Loss | Duplicates | Complexity | System Support |
|---|---|---|---|---|
| At-Most Once | Possible | Never | Low | Pub/Sub, Streams, Kafka |
| At-Least Once | Never | Possible | Medium | Streams + XCLAIM, Kafka |
| Exactly Once | Never | Never | High | Kafka (native), Streams (manual) |
